In [ ]:
import os
import sys
import glob
import pandas as pd

#avoid import errors
parent_dir=os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

from scripts.CDS_machine import CDS_2_gffcomp
import scripts.get_busco_db as bob
import scripts.get_genetic_code as ggc
from scripts.counting_machine import write_counts, CATEGORIES


In [ ]:
reference=["Cyanidioschyzon_merolae_strain_10D_280699"]
lyric=["Babesia_duncani_323732", "Emiliania_huxleyi_CCMP1516_280463", "Cyanidiococcus_yangmingshanensis_2690220", "Hydrurus_foetidus_2996", "Chlorella_sorokiniana_3076"]
isoquant=["Chlamydomonas_reinhardtii_3055"]
collection=reference+lyric+isoquant

raw_data="../../data/species"
query_email="wsxktjnfwcecxnotrf@gonrr.net"
busco_database="/no_backup/rg/references/busco_downloads"

!ls $raw_data

Babesia_duncani_323732
Chaetoceros_neogracilis_240364
Chlamydomonas_reinhardtii_3055
Cladocopium_goreaui_2562237
Conticribra_weissflogii_1577725
Cryptosporidium_parvum_Iowa_II_353152
Cyanidiococcus_yangmingshanensis_2690220
Cyanidioschyzon_merolae_strain_10d_280699
Cyanidioschyzon_merolae_strain_10D_280699
Cyanidium_caldarium_2771
Cylindrotheca_closterium_2856
Dunaliella_salina_3046
Durusdinium_trenchii_1381693
Eimeria_necatrix_51315
Emiliania_huxleyi_CCMP1516_280463
Entamoeba_histolytica_HM-1:IMSS_294381
Galdieria_yellowstonensis_3028027
Giardia_duodenalis_5741
Giardia_muris_5742
Haematococcus_lacustris_44745
Hamiltosporidium_tvaerminnensis_1176355
Leishmania_donovani_5661
Leishmania_infantum_JPCM5_435258
Micromonas_pusilla_CCMP1545_564608
Naegleria_gruberi_5762
Paramecium_bursaria_74790
Paramecium_tetraurelia_5888
Plasmodium_berghei_ANKA_5823
Plasmodium_falciparum_3D7_36329
Plasmodium_vivax_5855
Plasmodium_yoelii_5861
Symbiodinium_natans_878477
Symbiodinium_necroappetens_1628268
Tetr

# Run GeneID

In [ ]:
#geneid_command="time geneid -3P {parameter_file} {assembly} > {annotation.gff3}"

#units = <species>_<source>, taken from the trained parameter files
trained_units=[f.replace(".geneid.optimized.param", "")
               for f in os.listdir("../results/trainedParams")
               if f.endswith(".geneid.optimized.param")]
print(len(trained_units))

#checker(if things in file)
fileWritten=False
filepath="../job_commands/SARpred.txt"
with open(filepath, "w") as out:
    #create predictions folder command
    pred_folder="../results/pred"
    print(f"mkdir -p {pred_folder}", file=out)
    for unit in trained_units:
        sp=unit.rsplit("_", 1)[0]

        #shared cleaned genome for the species; model named by unit (source-tagged)
        ref=!realpath ../training_data/species/$sp/CLEAN_*.fna
        model=f"{pred_folder}/{unit}.gff3"

        #own-trained parameters for this unit
        param=os.path.abspath(f"../results/trainedParams/{unit}.geneid.optimized.param")
        fileWritten=True

        #geneid param fasta_assembly > gff_model
        command=f"time geneid -3P {param} {ref[0]} > {model}"

        #link inside the unit folder just in case
        logsDir_cmd=f"mkdir -p ../results/specie_logs/{unit}/" #if running from exsting parameters create logs folder
        lnPred_cmd=f"ln -vf {model} ../results/specie_logs/{unit}/"

        print(command)
        print(command, file=out)
        print(logsDir_cmd, file=out)
        print(lnPred_cmd, file=out)
        #in the end of the execution delete empty files
        print(f"find {pred_folder} -size 0 -print -delete", file=out)

if not fileWritten: #if the file is not written, delete it
    os.remove(filepath)

#Execute commands to predict CDS

In [1]:
!ls $pred_folder

1-GeneIDTrain.ipynb	  3-GeneIDPredict.ipynb
2-GeneIDGitCompare.ipynb  __pycache__


# Prediction analysis

## BUSCO assessment

In [ ]:
#agat for gff>prot
#evaluate every prediction file: <species>_<source>.gff3
pred_units=[os.path.basename(p)[:-len(".gff3")]
            for p in sorted(glob.glob("../results/pred/*.gff3"))]
print(len(pred_units))

#checker(if things in file)
fileWritten=False
filepath="../job_commands/buscoScoring.txt"
with open(filepath, "w") as f:
    print('cpus="${SLURM_CPUS_PER_TASK:-$(nproc)}"', file=f)
    print(f"mkdir -p ../results/summary/regular/busco_lineage ../results/summary/regular/busco_eukaryote", file=f)
    for unit in pred_units:
        try:
            sp=unit.rsplit("_", 1)[0]
            pred=f"../results/pred/{unit}.gff3"

            fileWritten=True

            #reference sequence (shared genome for the species)
            RefSeq=!realpath ../training_data/species/$sp/CLEAN_*.fna
            RefSeq=RefSeq[0]

            #create folder to store outputs
            prot_res=f"../results/specie_logs/{unit}/{unit}_prot.fa" #protein sequence output
            busco_outPath=f"../results/busco/" #busco comparisons output
            Lbusco_res=f"{unit}_Lbusco" #taxon-lineage busco out name
            Ebusco_res=f"{unit}_Ebusco" #eukaryota busco out name

            #get species taxon
            taxID=sp[sp.rfind("_")+1:]
            #find lineage
            busco_lineage=bob.taxon2lineage(taxID, query_email, f"{busco_database}/file_versions.tsv", "odb12")
            euk_lineage="eukaryota_odb12"
            #resolve the NCBI nuclear genetic code for this taxon (table 1 if unresolved)
            gcode=ggc.taxon2geneticcode(taxID, query_email) or 1

            #gff to protein command
            #per-task AGAT config so parallel array tasks don't collide on agat_config.yaml
            agat_cfg=f"agat_{unit}.yaml"
            expose_cmd=f"agat config --expose --output {agat_cfg} >/dev/null 2>&1"
            print(expose_cmd); print(expose_cmd, file=f)
            TOprotein_cmd=f"agat_sp_extract_sequences.pl -g {pred} -f {RefSeq} -t cds -p --table {gcode} --config {agat_cfg} -o {prot_res}"

            print(TOprotein_cmd)
            print(TOprotein_cmd, file=f)
            clean_cmd=f"rm -f {agat_cfg}"
            print(clean_cmd); print(clean_cmd, file=f)
            #deduplicate protein IDs (AGAT can emit duplicate names that break BUSCO)
            ND_prot_res=prot_res.replace(".fa", "_ND.fa")
            rename_cmd=f"seqkit rename -n {prot_res} > {ND_prot_res}"
            print(rename_cmd)
            print(rename_cmd, file=f)
            
            Lbusco_cmd=f"busco -m protein \
                        -i {ND_prot_res} \
                        --download_path {busco_database} \
                        -l {busco_lineage} \
                        --cpu $cpus \
                        -f \
                        --opt-out-run-stats \
                        --out_path {busco_outPath} \
                        -o {Lbusco_res}" #--offline #autolineage creates errors
            Lbusco_cmd=Lbusco_cmd.replace("                             ", " ")

            Ebusco_cmd=f"busco -m protein \
                        -i {ND_prot_res} \
                        --download_path {busco_database} \
                        -l {euk_lineage} \
                        --cpu $cpus \
                        -f \
                        --opt-out-run-stats \
                        --out_path {busco_outPath} \
                        -o {Ebusco_res}" #--offline #autolineage creates errors
            Ebusco_cmd=Ebusco_cmd.replace("                             ", " ")

            print(Lbusco_cmd); print(Lbusco_cmd, file=f)
            print(Ebusco_cmd); print(Ebusco_cmd, file=f)

            #collect each run JSON into its own summary folder (isoquant naming)
            print(f"ln -vf {busco_outPath}{Lbusco_res}/short_summary.specific.*.json ../results/summary/regular/busco_lineage/{Lbusco_res}.json", file=f)
            print(f"ln -vf {busco_outPath}{Ebusco_res}/short_summary.specific.*.json ../results/summary/regular/busco_eukaryote/{Ebusco_res}.json", file=f)

        except Exception as e:
            print(f"{unit} presents error: {e}")
            continue
        
if not fileWritten: #if the file is not written, delete it
    os.remove(filepath)
            

# Summary

The **regular** evaluation outputs are collected under `results/summary/regular/`: the busco cells above write the BUSCO summaries

In [ ]:
#metrics for the regular predictions -> results/summary/regular/counts/ per-species file + counts_summary.tsv.

results_dir = "../results"
summary_dir = f"{results_dir}/summary"
os.makedirs(summary_dir, exist_ok=True)

write_counts(results_dir, summary_dir, "regular", CATEGORIES["regular"])